# 01 - Walkthrough práctico: del radar a EarthFormer/CasCast

Este notebook cuenta una historia progresiva: primero entendemos el evento de radar, luego usamos persistencia como línea base, después miramos RMSE y CSI, y finalmente comparamos predicciones guardadas de EarthFormer y CasCast si existen.

La escala de colores usa umbrales discretos en mm/h: gris para 0-0.5, verde desde 0.5-2, y colores cálidos para lluvia más intensa.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from course_utils.data import (
    SAMPLE_FILES,
    LEAD_MINUTES,
    get_paths,
    load_sample,
    split_sequence,
    load_prediction,
    make_persistence_prediction,
)
from course_utils.metrics import (
    THRESHOLDS,
    THRESHOLD_LABELS,
    continuous_metrics,
    event_metrics_by_threshold_and_lead,
    summarize_events_by_threshold,
)
from course_utils.palette import apply_course_style, rain_cmap, RAIN_LEVELS
from course_utils.plotting import (
    plot_event_grid,
    plot_target_prediction_panel,
    plot_rmse_bias,
    plot_csi_by_threshold,
)

apply_course_style()
paths = get_paths(PROJECT_ROOT)
print(f"Proyecto: {paths.root}")
print(f"Muestras: {paths.sample_dir}")

## 1. Elegir un caso y mirar la tormenta

Cada archivo tiene 25 cuadros. Los primeros 13 son contexto observado; los últimos 12 son el futuro real.

In [ ]:
sample_name = SAMPLE_FILES[1]
sequence = load_sample(sample_name, paths)
inputs, target = split_sequence(sequence)
persistence = make_persistence_prediction(inputs)

print("Caso:", sample_name)
print("Entrada:", inputs.shape, "Target:", target.shape, "Persistencia:", persistence.shape)
print("Niveles de color mm/h:", RAIN_LEVELS)

In [ ]:
fig = plot_event_grid(inputs, target, persistence, sample_name, "persistencia")
plt.show()

## 2. Persistencia: una línea base honesta

Persistencia repite el último radar observado:

$$
\hat{Y}_{t+k} = X_t \quad \text{para } k=1,2,\dots,12
$$

Si la tormenta se desplaza, persistencia falla de una forma muy visual: conserva la forma, pero la deja en el lugar anterior.

## 3. RMSE y CSI

RMSE mide error promedio:

$$
\mathrm{RMSE}_k = \sqrt{\frac{1}{HW}\sum_{i=1}^{H}\sum_{j=1}^{W}(\hat{Y}_{k,i,j} - Y_{k,i,j})^2}
$$

CSI mide habilidad para eventos de lluvia por umbral:

$$
\mathrm{CSI} = \frac{TP}{TP + FP + FN}
$$

CSI ignora verdaderos negativos, que dominan mucho en campos de radar.

In [ ]:
cont = continuous_metrics(persistence, target)
events = event_metrics_by_threshold_and_lead(persistence, target, THRESHOLDS)
cont

In [ ]:
fig = plot_rmse_bias(cont)
plt.show()

fig = plot_csi_by_threshold(events)
plt.show()

In [ ]:
summarize_events_by_threshold(events)

## 4. Comparar modelos si hay predicciones guardadas

El curso reconoce estas carpetas:

- `outputs/predictions/persistence/`
- `outputs/predictions/earthformer/`
- `outputs/predictions/cascast/`
- `outputs/predictions/model/` como compatibilidad anterior.

Si EarthFormer o CasCast todavía no existen, la comparación se salta sin romper el notebook.

In [ ]:
def available_predictions_for_case(name):
    seq = load_sample(name, paths)
    x, y = split_sequence(seq)
    sources = {"persistence": make_persistence_prediction(x)}
    for label, folder in [("earthformer", paths.earthformer_dir), ("cascast", paths.cascast_dir), ("model", paths.model_dir)]:
        pred_path = folder / name
        if pred_path.exists():
            sources[label] = np.nan_to_num(np.load(pred_path).astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    return x, y, sources

inputs, target, predictions_by_model = available_predictions_for_case(sample_name)
print("Predicciones disponibles:", list(predictions_by_model))

In [ ]:
comparison_rows = []
for model_name, pred in predictions_by_model.items():
    c = continuous_metrics(pred, target)
    e = event_metrics_by_threshold_and_lead(pred, target, THRESHOLDS)
    comparison_rows.append({
        "model": model_name,
        "RMSE_mean": c["RMSE"].mean(),
        "MAE_mean": c["MAE"].mean(),
        "Bias_mean": c["Bias"].mean(),
        "CSI_0.5_mean": e[e["threshold_mm_h"] == 0.5]["CSI"].mean(),
        "CSI_5_mean": e[e["threshold_mm_h"] == 5.0]["CSI"].mean(),
        "CSI_10_mean": e[e["threshold_mm_h"] == 10.0]["CSI"].mean(),
    })

pd.DataFrame(comparison_rows).sort_values("RMSE_mean")

In [ ]:
for model_name, pred in predictions_by_model.items():
    fig = plot_event_grid(inputs, target, pred, sample_name, model_name, lead_indices=(2, 5, 8, 11))
    plt.show()

## 5. Fidelidad por archivo: todos los casos

Ahora resumimos cada archivo y cada modelo disponible. Esto permite preguntar: ¿qué casos son fáciles?, ¿qué casos pierden lluvia intensa?, ¿qué modelo mantiene mejor fidelidad espacial?

In [ ]:
all_rows = []
for name in SAMPLE_FILES:
    seq = load_sample(name, paths)
    x, y = split_sequence(seq)
    _, _, preds = available_predictions_for_case(name)
    for model_name, pred in preds.items():
        c = continuous_metrics(pred, y)
        e = event_metrics_by_threshold_and_lead(pred, y, THRESHOLDS)
        all_rows.append({
            "sample": name,
            "model": model_name,
            "RMSE_mean": c["RMSE"].mean(),
            "MAE_mean": c["MAE"].mean(),
            "Bias_mean": c["Bias"].mean(),
            "Pearson_mean": c["Pearson_r"].mean(),
            "CSI_0.5_mean": e[e["threshold_mm_h"] == 0.5]["CSI"].mean(),
            "CSI_2_mean": e[e["threshold_mm_h"] == 2.0]["CSI"].mean(),
            "CSI_5_mean": e[e["threshold_mm_h"] == 5.0]["CSI"].mean(),
            "CSI_10_mean": e[e["threshold_mm_h"] == 10.0]["CSI"].mean(),
        })

all_file_metrics = pd.DataFrame(all_rows)
all_file_metrics

In [ ]:
overall_metrics = (
    all_file_metrics
    .groupby("model", as_index=False)
    [["RMSE_mean", "MAE_mean", "Bias_mean", "Pearson_mean", "CSI_0.5_mean", "CSI_2_mean", "CSI_5_mean", "CSI_10_mean"]]
    .mean(numeric_only=True)
    .sort_values("RMSE_mean")
)
overall_metrics

In [ ]:
if len(all_file_metrics["model"].unique()) > 1:
    fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
    for model_name, group in all_file_metrics.groupby("model"):
        ax.scatter(group["RMSE_mean"], group["CSI_10_mean"], label=model_name, s=60)
    ax.set_xlabel("RMSE promedio (mm/h)")
    ax.set_ylabel("CSI promedio a 10 mm/h")
    ax.set_title("Fidelidad por caso: error promedio vs lluvia intensa")
    ax.legend()
    plt.show()
else:
    print("Solo hay un modelo disponible. Ejecuta inferencia para comparar EarthFormer/CasCast.")

## 6. Panel compacto de casos

Este panel muestra varios casos con la misma escala. Si existe CasCast, úsalo; si no, usa EarthFormer; si no, persistencia.

In [ ]:
preferred_model = "cascast" if (paths.cascast_dir / SAMPLE_FILES[0]).exists() else "earthformer" if (paths.earthformer_dir / SAMPLE_FILES[0]).exists() else "persistence"

cases = []
for name in SAMPLE_FILES[:3]:
    seq = load_sample(name, paths)
    x, y = split_sequence(seq)
    _, _, preds = available_predictions_for_case(name)
    pred = preds.get(preferred_model, preds["persistence"])
    cases.append({"label": name.split("_rain_rate")[0], "target": y, "prediction": pred})

fig = plot_target_prediction_panel(cases, title=f"Objetivo vs predicción: {preferred_model}")
plt.show()

## 7. Demo avanzado opcional: correr inferencia

Para producir predicciones de EarthFormer y CasCast usa el mismo ambiente del curso e instala extras:

```bash
conda activate nowcasting-course-lab
conda install -c pytorch -c nvidia pytorch=2.0.1 torchvision pytorch-cuda=11.8
pip install -r inference_requirements.txt
```

Luego prueba desde la línea de comandos. `--device auto` usa CUDA si existe y CPU si no:

```bash
python scripts/run_model_inference.py --stage earthformer --smoke --device auto
python scripts/run_model_inference.py --stage all --smoke --device auto
```

Si solo tienes CPU, prueba primero `--device cpu --ddim-steps 2 --cpu-threads 8`. Una corrida completa sobre los 5 casos sería:

```bash
python scripts/run_model_inference.py --stage all --sample all --ddim-steps 20 --device auto
python scripts/evaluate_predictions.py --sample all
```

El demo de abajo no corre automáticamente. Cambia `RUN_ADVANCED_MODEL_DEMO = True` solo si estás en GPU.

In [ ]:
from course_utils.model_inference import (
    ModelDemoUnavailable,
    check_model_demo_ready,
    load_model_demo_config,
    run_optional_model_demo,
)

config = load_model_demo_config(PROJECT_ROOT)
ok, message = check_model_demo_ready(config)
print(message)
print("CasCast repo:", config["cascast_repo"])
for key, value in config["checkpoints"].items():
    print(f"{key}: {value}")

In [ ]:
RUN_ADVANCED_MODEL_DEMO = False

if RUN_ADVANCED_MODEL_DEMO:
    try:
        outputs = run_optional_model_demo(
            sequence,
            sample_names=[sample_name],
            root=PROJECT_ROOT,
            ddim_steps=2,
            ensemble_member=1,
            cfg_weight=1.0,
            save_outputs=True,
        )
        print("EarthFormer:", outputs["earthformer"].shape)
        print("CasCast:", outputs["cascast"].shape)
    except ModelDemoUnavailable as exc:
        print("Demo no disponible:", exc)
else:
    print("Demo avanzado omitido. Cambia RUN_ADVANCED_MODEL_DEMO = True para ejecutarlo.")

## 8. Cierre conceptual

- Persistencia es el mínimo que un modelo debe superar.
- EarthFormer suele capturar movimiento y estructura de mesoescala.
- CasCast intenta recuperar detalle y realismo de pequeña escala usando difusión.
- RMSE y CSI juntos cuentan una historia mucho más útil que RMSE solo.